In [1]:
import pandas as pd
import os.path
import pickle
import numpy as np
from tensorflow import keras
from tensorflow.keras import utils

In [3]:
d=pd.read_csv(r"c:\Users\HP\Downloads\train.tsv", sep='\t')
d

,Unnamed: 0,title,text,subject,date,label
0,2619,Ex-CIA head says Trump remarks on Russia inter...,Former CIA director John Brennan on Friday cri...,politicsNews,"July 22, 2017",1
1,16043,YOU WON’T BELIEVE HIS PUNISHMENT! HISPANIC STO...,How did this man come to OWN this store? There...,Government News,"Jun 19, 2017",0
2,876,Federal Reserve governor Powell's policy views...,President Donald Trump on Thursday tapped Fede...,politicsNews,"November 2, 2017",1
3,19963,SCOUNDREL HILLARY SUPPORTER STARTS “TrumpLeaks...,Hillary Clinton ally David Brock is offering t...,left-news,"Sep 17, 2016",0
4,10783,NANCY PELOSI ARROGANTLY DISMISSES Questions on...,Pleading ignorance is a perfect ploy for Nancy...,politics,"May 26, 2017",0
...,...,...,...,...,...,...
29995,6880,U.S. aerospace industry urges Trump to help Ex...,The chief executive of the U.S. Aerospace Indu...,politicsNews,"December 6, 2016",1
29996,17818,Highlights: Hong Kong leader Carrie Lam delive...,The following are highlights of the maiden pol...,worldnews,"October 11, 2017",1
29997,5689,Obama Literally LAUGHS At Claims That Brexit M...,If there s one thing President Barack Obama is...,News,"June 28, 2016",0
29998,15805,Syrian army takes full control of Deir al-Zor ...,The Syrian army and its allies have taken full...,worldnews,"November 2, 2017",1


In [4]:
import time
from keras.callbacks import TensorBoard, CSVLogger
from keras.preprocessing import sequence
from keras.models import Sequential
from keras.layers import Dense,Flatten,LSTM,Conv1D,GlobalMaxPool1D,Dropout,Bidirectional
from keras.layers import Embedding
from keras import optimizers
from keras.layers import Input
from keras.models import Model
from tensorflow.keras.utils import plot_model
from IPython.display import SVG
from nltk.corpus import stopwords

In [5]:
print( d.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  30000 non-null  int64 
 1   title       30000 non-null  object
 2   text        30000 non-null  object
 3   subject     30000 non-null  object
 4   date        30000 non-null  object
 5   label       30000 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 1.4+ MB
None


In [6]:
d.shape

(30000, 6)

In [7]:
train_data = d.drop(d.columns[0], axis=1)

In [8]:
train_data

,title,text,subject,date,label
0,Ex-CIA head says Trump remarks on Russia inter...,Former CIA director John Brennan on Friday cri...,politicsNews,"July 22, 2017",1
1,YOU WON’T BELIEVE HIS PUNISHMENT! HISPANIC STO...,How did this man come to OWN this store? There...,Government News,"Jun 19, 2017",0
2,Federal Reserve governor Powell's policy views...,President Donald Trump on Thursday tapped Fede...,politicsNews,"November 2, 2017",1
3,SCOUNDREL HILLARY SUPPORTER STARTS “TrumpLeaks...,Hillary Clinton ally David Brock is offering t...,left-news,"Sep 17, 2016",0
4,NANCY PELOSI ARROGANTLY DISMISSES Questions on...,Pleading ignorance is a perfect ploy for Nancy...,politics,"May 26, 2017",0
...,...,...,...,...,...
29995,U.S. aerospace industry urges Trump to help Ex...,The chief executive of the U.S. Aerospace Indu...,politicsNews,"December 6, 2016",1
29996,Highlights: Hong Kong leader Carrie Lam delive...,The following are highlights of the maiden pol...,worldnews,"October 11, 2017",1
29997,Obama Literally LAUGHS At Claims That Brexit M...,If there s one thing President Barack Obama is...,News,"June 28, 2016",0
29998,Syrian army takes full control of Deir al-Zor ...,The Syrian army and its allies have taken full...,worldnews,"November 2, 2017",1


In [9]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    30000 non-null  object
 1   text     30000 non-null  object
 2   subject  30000 non-null  object
 3   date     30000 non-null  object
 4   label    30000 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.1+ MB


In [10]:
train_data.duplicated().sum()

92

In [11]:
train_data.drop_duplicates(inplace=True)

In [12]:
train_data.duplicated().sum()

0

In [13]:
pip install textblob scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
df=pd.read_csv(r"c:\Users\HP\Downloads\test.tsv", sep='\t')
df

,Unnamed: 0,title,text,subject,date,label
0,8104,Conservatives Will HATE What Donald Trump Just...,Donald Trump isn t exactly a stranger to makin...,News,"February 14, 2016",0
1,7467,Trump victory may create new tension between U...,Donald Trump’s U.S. election victory may creat...,politicsNews,"November 9, 2016",1
2,9473,WATCH: Hundreds of ILLEGAL ALIENS Storm Senate...,A couple of quick questions come to mind when ...,politics,"Nov 9, 2017",0
3,276,"Democratic Senator Franken to resign: CNN, cit...",U.S. Democratic Senator Al Franken will announ...,politicsNews,"December 7, 2017",1
4,19274,GANG OF DOMESTIC TERRORISTS Violently Attack L...,***WARNING*** Violence is graphic***This Trump...,left-news,"Jan 21, 2017",0
...,...,...,...,...,...,...
8262,5469,Russian MP says Flynn was forced to resign to ...,A senior Russian lawmaker said on Tuesday it w...,politicsNews,"February 14, 2017",1
8263,5079,Highlights: The Trump presidency on March 7 at...,Highlights of the day for U.S. President Donal...,politicsNews,"March 7, 2017",1
8264,20425,SHOCKER! WAS MUSLIM TERRORIST GAY? Used Gay Da...,"Of course, Mateen s Muslim father vehemently d...",left-news,"Jun 13, 2016",0
8265,22063,John McCain and The Cancer of Conflict,Patrick Henningsen 21st Century WireThis week ...,US_News,"July 21, 2017",0


In [3]:
test_data = df.drop(df.columns[0], axis=1)
test_data

,title,text,subject,date,label
0,Conservatives Will HATE What Donald Trump Just...,Donald Trump isn t exactly a stranger to makin...,News,"February 14, 2016",0
1,Trump victory may create new tension between U...,Donald Trump’s U.S. election victory may creat...,politicsNews,"November 9, 2016",1
2,WATCH: Hundreds of ILLEGAL ALIENS Storm Senate...,A couple of quick questions come to mind when ...,politics,"Nov 9, 2017",0
3,"Democratic Senator Franken to resign: CNN, cit...",U.S. Democratic Senator Al Franken will announ...,politicsNews,"December 7, 2017",1
4,GANG OF DOMESTIC TERRORISTS Violently Attack L...,***WARNING*** Violence is graphic***This Trump...,left-news,"Jan 21, 2017",0
...,...,...,...,...,...
8262,Russian MP says Flynn was forced to resign to ...,A senior Russian lawmaker said on Tuesday it w...,politicsNews,"February 14, 2017",1
8263,Highlights: The Trump presidency on March 7 at...,Highlights of the day for U.S. President Donal...,politicsNews,"March 7, 2017",1
8264,SHOCKER! WAS MUSLIM TERRORIST GAY? Used Gay Da...,"Of course, Mateen s Muslim father vehemently d...",left-news,"Jun 13, 2016",0
8265,John McCain and The Cancer of Conflict,Patrick Henningsen 21st Century WireThis week ...,US_News,"July 21, 2017",0


In [16]:
test_data.duplicated().sum()

10

In [4]:
test_data.drop_duplicates(inplace=True)

In [18]:
test_data.duplicated().sum()

0

data preprocessing

In [20]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import re

# Define clickbait and sensational keywords
clickbait_keywords = [
    'shocking', 'exclusive', 'breaking', 'urgent', 'warning',
    'incredible', 'unbelievable', 'miracle', 'secret', 'banned',
    'revolutionary', 'mind-blowing', 'you won\'t believe',
    'sensational', 'amazing', 'stunning', 'viral', 'exposed',
    'jaw-dropping', 'must see', 'conspiracy', 'scandal'
]

def calculate_keyword_density(text, keywords):
    """
    Calculate the density of specified keywords in the text
    """
    if pd.isna(text):
        return 0

    text = text.lower()
    word_count = len(text.split())
    if word_count == 0:
        return 0

    keyword_count = sum(text.count(keyword.lower()) for keyword in keywords)
    return keyword_count / word_count

def create_features(df, tfidf_title=None, tfidf_text=None, training=False):
    """
    Create features from the text and title columns
    Parameters:
        df: Input DataFrame
        tfidf_title: TfidfVectorizer for title (if None, will create new one)
        tfidf_text: TfidfVectorizer for text (if None, will create new one)
        training: Boolean indicating if this is for training data
    """
    features = pd.DataFrame(index=df.index)

    # Calculate keyword densities
    features['title_keyword_density'] = df['title'].apply(
        lambda x: calculate_keyword_density(x, clickbait_keywords)
    )

    features['text_keyword_density'] = df['text'].apply(
        lambda x: calculate_keyword_density(x, clickbait_keywords)
    )

    # Create or use existing TF-IDF vectorizers
    if training:
        tfidf_title = TfidfVectorizer(max_features=1000, stop_words='english')
        tfidf_text = TfidfVectorizer(max_features=2000, stop_words='english')
        title_features = tfidf_title.fit_transform(df['title'].fillna(''))
        text_features = tfidf_text.fit_transform(df['text'].fillna(''))
    else:
        title_features = tfidf_title.transform(df['title'].fillna(''))
        text_features = tfidf_text.transform(df['text'].fillna(''))

    # Convert to DataFrame
    title_df = pd.DataFrame(
        title_features.toarray(),
        columns=[f'title_tfidf_{i}' for i in range(title_features.shape[1])],
        index=df.index
    )

    text_df = pd.DataFrame(
        text_features.toarray(),
        columns=[f'text_tfidf_{i}' for i in range(text_features.shape[1])],
        index=df.index
    )

    # Combine all features
    features = pd.concat([features, title_df, text_df], axis=1)

    if training:
        return features, tfidf_title, tfidf_text
    return features

def main():
    # Load your data
    # Assuming train_data and test_data are your DataFrames
    # train_data = pd.read_csv('train.csv')
    # test_data = pd.read_csv('test.csv')

    # Create features for training data
    X_train, tfidf_title_vectorizer, tfidf_text_vectorizer = create_features(
        train_data, training=True
    )
    y_train = train_data['label']  # Assuming 'label' is your target column

    # Verify shapes match
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")

    # Train the model
    rf_classifier = RandomForestClassifier(n_estimators=200, random_state=42)
    rf_classifier.fit(X_train, y_train)

    # Process test data
    X_test = create_features(
        test_data,
        tfidf_title_vectorizer,
        tfidf_text_vectorizer,
        training=False
    )
    
    # Make predictions
    predictions = rf_classifier.predict(X_test)

    # If test data has labels, evaluate the model
    if 'label' in test_data.columns:
        print("\nModel Performance on Test Data:")
        print(classification_report(test_data['label'], predictions))
        print("\nConfusion Matrix:")
        print(confusion_matrix(test_data['label'], predictions))

    # Add predictions to test data
    test_data['predicted_label'] = predictions

    # Feature importance analysis
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': rf_classifier.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10))

    # Save predictions
    test_data.to_csv('predictions.csv', index=False)

if __name__ == "__main__":
    main()

X_train shape: (29908, 3002)
y_train shape: (29908,)

Model Performance on Test Data:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4284
           1       0.98      0.99      0.99      3973

    accuracy                           0.99      8257
   macro avg       0.99      0.99      0.99      8257
weighted avg       0.99      0.99      0.99      8257


Confusion Matrix:
[[4222   62]
 [  51 3922]]

Top 10 Most Important Features:
              feature  importance
2589  text_tfidf_1587    0.066395
944   title_tfidf_942    0.051146
1683   text_tfidf_681    0.028988
1872   text_tfidf_870    0.027981
1979   text_tfidf_977    0.019316
2551  text_tfidf_1549    0.017800
1387   text_tfidf_385    0.016281
2050  text_tfidf_1048    0.015577
1760   text_tfidf_758    0.013423
2943  text_tfidf_1941    0.013110


In [21]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pickle
import os
import re

# Define clickbait and sensational keywords
clickbait_keywords = [
    'shocking', 'exclusive', 'breaking', 'urgent', 'warning',
    'incredible', 'unbelievable', 'miracle', 'secret', 'banned',
    'revolutionary', 'mind-blowing', 'you won\'t believe',
    'sensational', 'amazing', 'stunning', 'viral', 'exposed',
    'jaw-dropping', 'must see', 'conspiracy', 'scandal'
]

def calculate_keyword_density(text, keywords):
    """
    Calculate the density of specified keywords in the text
    """
    if pd.isna(text):
        return 0

    text = text.lower()
    word_count = len(text.split())
    if word_count == 0:
        return 0

    keyword_count = sum(text.count(keyword.lower()) for keyword in keywords)
    return keyword_count / word_count

def create_features(df, tfidf_title=None, tfidf_text=None, training=False):
    """
    Create features from the text and title columns
    Parameters:
        df: Input DataFrame
        tfidf_title: TfidfVectorizer for title (if None, will create new one)
        tfidf_text: TfidfVectorizer for text (if None, will create new one)
        training: Boolean indicating if this is for training data
    """
    features = pd.DataFrame(index=df.index)

    # Calculate keyword densities
    features['title_keyword_density'] = df['title'].apply(
        lambda x: calculate_keyword_density(x, clickbait_keywords)
    )

    features['text_keyword_density'] = df['text'].apply(
        lambda x: calculate_keyword_density(x, clickbait_keywords)
    )

    # Create or use existing TF-IDF vectorizers
    if training:
        tfidf_title = TfidfVectorizer(max_features=1000, stop_words='english')
        tfidf_text = TfidfVectorizer(max_features=2000, stop_words='english')
        title_features = tfidf_title.fit_transform(df['title'].fillna(''))
        text_features = tfidf_text.fit_transform(df['text'].fillna(''))
    else:
        title_features = tfidf_title.transform(df['title'].fillna(''))
        text_features = tfidf_text.transform(df['text'].fillna(''))

    # Convert to DataFrame
    title_df = pd.DataFrame(
        title_features.toarray(),
        columns=[f'title_tfidf_{i}' for i in range(title_features.shape[1])],
        index=df.index
    )

    text_df = pd.DataFrame(
        text_features.toarray(),
        columns=[f'text_tfidf_{i}' for i in range(text_features.shape[1])],
        index=df.index
    )

    # Combine all features
    features = pd.concat([features, title_df, text_df], axis=1)

    if training:
        return features, tfidf_title, tfidf_text
    return features

def save_models(rf_classifier, tfidf_title_vectorizer, tfidf_text_vectorizer, model_dir='models'):
    """
    Save the trained models to disk
    """
    # Create models directory if it doesn't exist
    os.makedirs(model_dir, exist_ok=True)
    
    # Save the Random Forest classifier
    with open(os.path.join(model_dir, 'rf_classifier.pkl'), 'wb') as f:
        pickle.dump(rf_classifier, f)
    
    # Save the TF-IDF vectorizers
    with open(os.path.join(model_dir, 'tfidf_title_vectorizer.pkl'), 'wb') as f:
        pickle.dump(tfidf_title_vectorizer, f)
        
    with open(os.path.join(model_dir, 'tfidf_text_vectorizer.pkl'), 'wb') as f:
        pickle.dump(tfidf_text_vectorizer, f)

def load_models(model_dir='models'):
    """
    Load the trained models from disk
    """
    # Load the Random Forest classifier
    with open(os.path.join(model_dir, 'rf_classifier.pkl'), 'rb') as f:
        rf_classifier = pickle.load(f)
    
    # Load the TF-IDF vectorizers
    with open(os.path.join(model_dir, 'tfidf_title_vectorizer.pkl'), 'rb') as f:
        tfidf_title_vectorizer = pickle.load(f)
        
    with open(os.path.join(model_dir, 'tfidf_text_vectorizer.pkl'), 'rb') as f:
        tfidf_text_vectorizer = pickle.load(f)
        
    return rf_classifier, tfidf_title_vectorizer, tfidf_text_vectorizer

def main():
    # Load your data
    # Assuming train_data and test_data are your DataFrames
    # train_data = pd.read_csv('train.csv')
    # test_data = pd.read_csv('test.csv')

    # Create features for training data
    X_train, tfidf_title_vectorizer, tfidf_text_vectorizer = create_features(
        train_data, training=True
    )
    y_train = train_data['label']  # Assuming 'label' is your target column

    # Verify shapes match
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")

    # Train the model
    rf_classifier = RandomForestClassifier(n_estimators=200, random_state=42)
    rf_classifier.fit(X_train, y_train)

    # Save the trained models
    save_models(rf_classifier, tfidf_title_vectorizer, tfidf_text_vectorizer)
    print("Models saved successfully!")

    # Process test data
    X_test = create_features(
        test_data,
        tfidf_title_vectorizer,
        tfidf_text_vectorizer,
        training=False
    )
    
    # Make predictions
    predictions = rf_classifier.predict(X_test)

    # If test data has labels, evaluate the model
    if 'label' in test_data.columns:
        print("\nModel Performance on Test Data:")
        print(classification_report(test_data['label'], predictions))
        print("\nConfusion Matrix:")
        print(confusion_matrix(test_data['label'], predictions))

    # Add predictions to test data
    test_data['predicted_label'] = predictions

    # Feature importance analysis
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': rf_classifier.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10))

    # Save predictions
    test_data.to_csv('predictions.csv', index=False)

if __name__ == "__main__":
    main()

X_train shape: (29908, 3002)
y_train shape: (29908,)
Models saved successfully!

Model Performance on Test Data:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4284
           1       0.98      0.99      0.99      3973

    accuracy                           0.99      8257
   macro avg       0.99      0.99      0.99      8257
weighted avg       0.99      0.99      0.99      8257


Confusion Matrix:
[[4222   62]
 [  51 3922]]

Top 10 Most Important Features:
              feature  importance
2589  text_tfidf_1587    0.066395
944   title_tfidf_942    0.051146
1683   text_tfidf_681    0.028988
1872   text_tfidf_870    0.027981
1979   text_tfidf_977    0.019316
2551  text_tfidf_1549    0.017800
1387   text_tfidf_385    0.016281
2050  text_tfidf_1048    0.015577
1760   text_tfidf_758    0.013423
2943  text_tfidf_1941    0.013110


In [6]:
import pickle
import os

def load_models(model_dir='models'):
    """
    Load the trained models from disk
    """
    # Load the Random Forest classifier
    with open(os.path.join(model_dir, 'rf_classifier.pkl'), 'rb') as f:
        rf_classifier = pickle.load(f)

    # Load the TF-IDF vectorizers
    with open(os.path.join(model_dir, 'tfidf_title_vectorizer.pkl'), 'rb') as f:
        tfidf_title_vectorizer = pickle.load(f)

    with open(os.path.join(model_dir, 'tfidf_text_vectorizer.pkl'), 'rb') as f:
        tfidf_text_vectorizer = pickle.load(f)

    return rf_classifier, tfidf_title_vectorizer, tfidf_text_vectorizer


In [12]:
def calculate_keyword_density(text, keywords):
    """
    Calculate the density of specified keywords in the text
    """
    if pd.isna(text):
        return 0

    text = text.lower()
    word_count = len(text.split())
    if word_count == 0:
        return 0

    keyword_count = sum(text.count(keyword.lower()) for keyword in keywords)
    return keyword_count / word_count


In [13]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import pickle

# Load models (this assumes that 'load_models' and model files are correct)
def load_models(model_dir='models'):
    with open(os.path.join(model_dir, 'rf_classifier.pkl'), 'rb') as f:
        rf_classifier = pickle.load(f)

    with open(os.path.join(model_dir, 'tfidf_title_vectorizer.pkl'), 'rb') as f:
        tfidf_title_vectorizer = pickle.load(f)

    with open(os.path.join(model_dir, 'tfidf_text_vectorizer.pkl'), 'rb') as f:
        tfidf_text_vectorizer = pickle.load(f)

    return rf_classifier, tfidf_title_vectorizer, tfidf_text_vectorizer

# Define clickbait and sensational keywords
clickbait_keywords = [
    'shocking', 'exclusive', 'breaking', 'urgent', 'warning',
    'incredible', 'unbelievable', 'miracle', 'secret', 'banned',
    'revolutionary', 'mind-blowing', 'you won\'t believe',
    'sensational', 'amazing', 'stunning', 'viral', 'exposed',
    'jaw-dropping', 'must see', 'conspiracy', 'scandal'
]

def calculate_keyword_density(text, keywords):
    """
    Calculate the density of specified keywords in the text
    """
    if pd.isna(text):
        return 0

    text = text.lower()
    word_count = len(text.split())
    if word_count == 0:
        return 0

    keyword_count = sum(text.count(keyword.lower()) for keyword in keywords)
    return keyword_count / word_count

def create_features(df, tfidf_title=None, tfidf_text=None, training=False):
    """
    Create features from the text and title columns
    """
    features = pd.DataFrame(index=df.index)

    # Calculate keyword densities
    features['title_keyword_density'] = df['title'].apply(
        lambda x: calculate_keyword_density(x, clickbait_keywords)
    )

    features['text_keyword_density'] = df['text'].apply(
        lambda x: calculate_keyword_density(x, clickbait_keywords)
    )

    # Create or use existing TF-IDF vectorizers
    if training:
        tfidf_title = TfidfVectorizer(max_features=1000, stop_words='english')
        tfidf_text = TfidfVectorizer(max_features=2000, stop_words='english')
        title_features = tfidf_title.fit_transform(df['title'].fillna(''))
        text_features = tfidf_text.fit_transform(df['text'].fillna(''))
    else:
        title_features = tfidf_title.transform(df['title'].fillna(''))
        text_features = tfidf_text.transform(df['text'].fillna(''))

    # Convert to DataFrame
    title_df = pd.DataFrame(
        title_features.toarray(),
        columns=[f'title_tfidf_{i}' for i in range(title_features.shape[1])],
        index=df.index
    )

    text_df = pd.DataFrame(
        text_features.toarray(),
        columns=[f'text_tfidf_{i}' for i in range(text_features.shape[1])],
        index=df.index
    )

    # Combine all features
    features = pd.concat([features, title_df, text_df], axis=1)

    if training:
        return features, tfidf_title, tfidf_text
    return features

# Load the models
rf_classifier, tfidf_title_vectorizer, tfidf_text_vectorizer = load_models(model_dir='models')

# Assuming 'test_data' is already loaded and contains a 'label' column
# Generate features for the test data
X_test = create_features(test_data, tfidf_title=tfidf_title_vectorizer, tfidf_text=tfidf_text_vectorizer)

# Ground truth labels
y_true = test_data['label']

# Predict labels using the loaded Random Forest classifier
y_pred = rf_classifier.predict(X_test)

# Display confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

# Display the classification report for detailed metrics
print("\nClassification Report:\n", classification_report(y_true, y_pred))


Confusion Matrix:
 [[4222   62]
 [  51 3922]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      4284
           1       0.98      0.99      0.99      3973

    accuracy                           0.99      8257
   macro avg       0.99      0.99      0.99      8257
weighted avg       0.99      0.99      0.99      8257

